In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mlg-ulb/creditcardfraud")

print("Path to dataset files:", path)

Path to dataset files: /kaggle/input/creditcardfraud


In [2]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import DBSCAN
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns


In [3]:
data=pd.read_csv(path+"/creditcard.csv")
data.head()

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0


In [4]:
print("Dataset Info:")
print(data.info())
print("\nFirst 5 rows:")
print(data.head())
print("\nClass distribution:")
print(data['Class'].value_counts())


Dataset Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 284807 entries, 0 to 284806
Data columns (total 31 columns):
 #   Column  Non-Null Count   Dtype  
---  ------  --------------   -----  
 0   Time    284807 non-null  float64
 1   V1      284807 non-null  float64
 2   V2      284807 non-null  float64
 3   V3      284807 non-null  float64
 4   V4      284807 non-null  float64
 5   V5      284807 non-null  float64
 6   V6      284807 non-null  float64
 7   V7      284807 non-null  float64
 8   V8      284807 non-null  float64
 9   V9      284807 non-null  float64
 10  V10     284807 non-null  float64
 11  V11     284807 non-null  float64
 12  V12     284807 non-null  float64
 13  V13     284807 non-null  float64
 14  V14     284807 non-null  float64
 15  V15     284807 non-null  float64
 16  V16     284807 non-null  float64
 17  V17     284807 non-null  float64
 18  V18     284807 non-null  float64
 19  V19     284807 non-null  float64
 20  V20     284807 non-null  float64
 

In [5]:
print("\nMissing values:")
print(data.isnull().sum())


Missing values:
Time      0
V1        0
V2        0
V3        0
V4        0
V5        0
V6        0
V7        0
V8        0
V9        0
V10       0
V11       0
V12       0
V13       0
V14       0
V15       0
V16       0
V17       0
V18       0
V19       0
V20       0
V21       0
V22       0
V23       0
V24       0
V25       0
V26       0
V27       0
V28       0
Amount    0
Class     0
dtype: int64


In [6]:
X = data.drop(columns=['Time', 'Class'])
y = data['Class']

In [7]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled = pd.DataFrame(X_scaled, columns=X.columns)


In [8]:
X_scaled.to_csv('X_scaled.csv', index=False)
np.save('y_labels.npy', y)


In [9]:
print("\nPreprocessing complete. Scaled features saved as 'X_scaled.csv'.")



Preprocessing complete. Scaled features saved as 'X_scaled.csv'.


In [10]:
plt.figure(figsize=(10, 6))
sns.histplot(data[data['Class'] == 0]['Amount'], bins=50, color='blue', label='Normal', alpha=0.5)
sns.histplot(data[data['Class'] == 1]['Amount'], bins=50, color='red', label='Fraud', alpha=0.5)
plt.legend()
plt.title('Transaction Amount Distribution (Normal vs Fraud)')
plt.xlabel('Amount')
plt.ylabel('Frequency')
plt.savefig('amount_distribution.png')
plt.close()

In [11]:
plt.figure(figsize=(10, 6))
sns.histplot(data[data['Class'] == 0]['V1'], bins=50, color='blue', label='Normal', alpha=0.5)
sns.histplot(data[data['Class'] == 1]['V1'], bins=50, color='red', label='Fraud', alpha=0.5)
plt.legend()
plt.title('V1 Feature Distribution (Normal vs Fraud)')
plt.xlabel('V1')
plt.ylabel('Frequency')
plt.savefig('v1_distribution.png')
plt.close()

print("Feature visualizations saved as 'amount_distribution.png' and 'v1_distribution.png'.")


Feature visualizations saved as 'amount_distribution.png' and 'v1_distribution.png'.


In [12]:
X_scaled = pd.read_csv('X_scaled.csv')
y = np.load('y_labels.npy')


In [ ]:
print("\nApplying DBSCAN for anomaly detection...")
dbscan = DBSCAN(eps=3.0, min_samples=10)  # eps and min_samples may need tuning
clusters = dbscan.fit_predict(X_scaled)



Applying DBSCAN for anomaly detection...


In [ ]:
dbscan_labels = np.where(clusters == -1, 1, 0)


Applying DBSCAN for anomaly detection...


In [ ]:
dbscan_labels = np.where(clusters == -1, 1, 0)


In [ ]:
print("\nDBSCAN Results:")
print(classification_report(y, dbscan_labels))
print("\nConfusion Matrix:")
print(confusion_matrix(y, dbscan_labels))


In [ ]:
anomaly_count = np.sum(dbscan_labels == 1)
normal_count = np.sum(dbscan_labels == 0)
plt.figure(figsize=(8, 6))
sns.barplot(x=['Normal', 'Anomaly'], y=[normal_count, anomaly_count], palette='Set2')
plt.title('DBSCAN: Number of Normal vs Anomalous Transactions')
plt.ylabel('Count')
plt.savefig('dbscan_anomaly_counts.png')
plt.close()

print("DBSCAN anomaly detection complete. Results plot saved as 'dbscan_anomaly_counts.png'.")